In [14]:
import keras
from keras import layers, models
from keras.layers import Dropout, Flatten, MaxPooling2D, Conv2D
import numpy as np
import tensorflow as tf
%run data_preprocessing_01.ipynb

 Class Encoding: {'COVID': 0, 'Lung_Opacity': 1, 'Normal': 2, 'Viral Pneumonia': 3}
Data Splitted -> Train: 25398 | Val: 8466 | Test: 8466
DataLoaders are ready for CNN & FFNN!


In [ ]:
# CNN
# Sources: Ma3ref Courses: Machine Learning.
# CONV2D: extracts features
# Maxpooling: feature selection and size reduction
# Flattening: converts to 1D
# Dropout: reduces overfitting by randomly disabling neurons.

# Hyperparameters tuned for 128x128x1 input
model = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 1)), # Hidden layers
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dropout(0.5), # dropout by 0.5
    layers.Dense(128, activation='relu'), # extra hidden layer
    layers.Dense(4, activation='softmax') # Output layer adjusted for 4 classes
])

c:\Users\DREAMS\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:

# Compile the model
model.compile(
    optimizer="Adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary() # summarize model performance.

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,580 (12.61 MB)

 Trainable params: 3,304,580 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Helper to convert PyTorch DataLoader to TensorFlow Dataset
def torch_to_tf(dataloader):
    def generator():
        for images, labels in dataloader:
            # Convert (Batch, Channel, Height, Width) -> (Batch, Height, Width, Channel)
            yield images.permute(0, 2, 3, 1).numpy(), labels.numpy()
    
    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, 128, 128, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    ).prefetch(tf.data.AUTOTUNE)


In [18]:
# Convert loaders (Ensure cell q-6yN2p4RvPC ran successfully first)
tf_train_ds = torch_to_tf(train_loader)
tf_val_ds = torch_to_tf(val_loader)

# Training using model.fit
EPOCHS = 5
history = model.fit(
    tf_train_ds,
    validation_data=tf_val_ds,
    epochs=EPOCHS
)

Epoch 1/5
    794/Unknown 221s 274ms/step - accuracy: 0.5872 - loss: 0.9725

c:\Users\DREAMS\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


794/794 ━━━━━━━━━━━━━━━━━━━━ 273s 340ms/step - accuracy: 0.6363 - loss: 0.8683 - val_accuracy: 0.7108 - val_loss: 0.7125
Epoch 2/5
794/794 ━━━━━━━━━━━━━━━━━━━━ 254s 319ms/step - accuracy: 0.7086 - loss: 0.7101 - val_accuracy: 0.6968 - val_loss: 0.7148
Epoch 3/5
794/794 ━━━━━━━━━━━━━━━━━━━━ 268s 337ms/step - accuracy: 0.7375 - loss: 0.6526 - val_accuracy: 0.7644 - val_loss: 0.5954
Epoch 4/5
794/794 ━━━━━━━━━━━━━━━━━━━━ 223s 281ms/step - accuracy: 0.7522 - loss: 0.6092 - val_accuracy: 0.7726 - val_loss: 0.5696
Epoch 5/5
794/794 ━━━━━━━━━━━━━━━━━━━━ 197s 247ms/step - accuracy: 0.7665 - loss: 0.5810 - val_accuracy: 0.7795 - val_loss: 0.5561
